# 📊 Plotly Introduction — Full Notebook with Solutions

This notebook introduces **Plotly** for interactive data visualization.

We will use the `tips` dataset and learn:

- Plotly Express (`px`) for fast interactive plots
- Plotly Graph Objects (`go`) for manual/custom plots
- Scatter plots, bar charts, box plots, histograms, heatmaps, bubble charts, 3D plots
- How to aggregate data before plotting
- How to customize titles, labels, hover data, and layout

Each exercise contains:
1. A clear request
2. A short explanation
3. A solved code cell

## 1. Import Libraries

We need:

- `pandas` for data manipulation
- `seaborn` only to load the sample dataset
- `plotly.express` for easy plotting
- `plotly.graph_objects` for manual plotting

In [1]:
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

## 2. Load the Dataset

### Request
Load the `tips` dataset and display the first rows.

### Explanation
The `tips` dataset contains restaurant bills, tips, customer gender, smoker status, day, time, and group size.

In [2]:
df = sns.load_dataset("tips")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 3. Basic Dataset Exploration

### Request
Check the shape, columns, data types, and missing values.

### Explanation
Before plotting, we need to understand the structure of the dataset.

In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Shape: (244, 7)

Columns:
['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']

Data types:
total_bill     float64
tip            float64
sex           category
smoker        category
day           category
time          category
size             int64
dtype: object

Missing values:
total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64


## 4. Identify Numerical and Categorical Columns

### Request
Separate numerical and categorical columns.

### Explanation
This helps us choose correct chart types.

- Numerical columns are used in scatter plots, histograms, box plots, correlations.
- Categorical columns are used for grouping, colors, count plots, and comparisons.

Important: Some categorical columns may have dtype `category`, not `object`.

In [4]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns

print("Numerical columns:", numerical_cols.tolist())
print("Categorical columns:", categorical_cols.tolist())

Numerical columns: ['total_bill', 'tip', 'size']
Categorical columns: ['sex', 'smoker', 'day', 'time']


## 5. Value Counts for Categorical Columns

### Request
Show how many records exist in each category.

### Explanation
This helps us understand the distribution of categorical variables such as `day`, `sex`, `smoker`, and `time`.

In [5]:
for col in categorical_cols:
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


sex value counts:
sex
Male      157
Female     87
Name: count, dtype: int64

smoker value counts:
smoker
No     151
Yes     93
Name: count, dtype: int64

day value counts:
day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64

time value counts:
time
Dinner    176
Lunch      68
Name: count, dtype: int64


# Part A — Plotly Express

Plotly Express is the easiest way to create interactive charts.
Usually, one line of code is enough.

## 6. Scatter Plot: Total Bill vs Tip

### Request
Create an interactive scatter plot showing the relationship between `total_bill` and `tip`.

### Explanation
Each point represents one restaurant bill.
This chart helps us see whether bigger bills usually lead to bigger tips.

In [6]:
fig = px.scatter(
    data_frame=df,
    x="total_bill",
    y="tip",
    title="Total Bill vs Tip"
)

fig.show()

## 7. Scatter Plot with Color

### Request
Create the same scatter plot, but color the points by `day`.

### Explanation
The `color` argument works similarly to `hue` in Seaborn.
It helps compare groups inside the same plot.

In [7]:
fig = px.scatter(
    data_frame=df,
    x="total_bill",
    y="tip",
    color="day",
    title="Total Bill vs Tip by Day"
)

fig.show()

## 8. Scatter Plot with Extra Hover Information

### Request
Add extra information to the hover tooltip.

### Explanation
Plotly is interactive. We can decide which columns appear when we hover over a point.

In [8]:
fig = px.scatter(
    data_frame=df,
    x="total_bill",
    y="tip",
    color="day",
    hover_data=["sex", "smoker", "time", "size"],
    title="Total Bill vs Tip with Hover Details"
)

fig.show()

## 9. Facet Plot: Separate Charts by Smoker Status

### Request
Create a scatter plot split into separate columns by `smoker`.

### Explanation
`facet_col` creates multiple small charts.
This is useful when we want to compare categories separately.

In [9]:
fig = px.scatter(
    data_frame=df,
    x="total_bill",
    y="tip",
    color="day",
    facet_col="smoker",
    title="Total Bill vs Tip Split by Smoker Status"
)

fig.show()

## 10. Bar Chart Without Aggregation

### Request
Create a bar chart with `day` on the x-axis and `total_bill` on the y-axis.

### Explanation
Important: `px.bar()` does **not automatically calculate the mean**.
It plots row-level values. If many rows have the same day, Plotly stacks/overlays many values.

In [10]:
fig = px.bar(
    data_frame=df,
    x="day",
    y="total_bill",
    title="Total Bill by Day - Row Level Values"
)

fig.show()

## 11. Bar Chart with Mean Total Bill per Day

### Request
Calculate the average total bill per day, then plot it.

### Explanation
For summary bar charts, first use `groupby()`, then plot the aggregated result.

In [11]:
mean_bill_by_day = (
    df.groupby("day", observed=False)["total_bill"]
      .mean()
      .reset_index()
)

mean_bill_by_day

,day,total_bill
0,Thur,17.682742
1,Fri,17.151579
2,Sat,20.441379
3,Sun,21.410000


In [12]:
fig = px.bar(
    data_frame=mean_bill_by_day,
    x="day",
    y="total_bill",
    title="Average Total Bill per Day",
    labels={"total_bill": "Average Total Bill", "day": "Day"}
)

fig.show()

## 12. Grouped Bar Chart: Average Bill by Day and Smoker

### Request
Show the average total bill for smokers and non-smokers for each day.

### Explanation
This uses two concepts:
1. `groupby(["day", "smoker"])`
2. `barmode="group"` to show bars side by side

In [13]:
mean_bill_day_smoker = (
    df.groupby(["day", "smoker"], observed=False)["total_bill"]
      .mean()
      .reset_index()
)

mean_bill_day_smoker

,day,smoker,total_bill
0,Thur,Yes,19.190588
1,Thur,No,17.113111
2,Fri,Yes,16.813333
3,Fri,No,18.420000
4,Sat,Yes,21.276667
5,Sat,No,19.661778
6,Sun,Yes,24.120000
7,Sun,No,20.506667


In [14]:
fig = px.bar(
    data_frame=mean_bill_day_smoker,
    x="day",
    y="total_bill",
    color="smoker",
    barmode="group",
    title="Average Total Bill by Day and Smoker",
    labels={"total_bill": "Average Total Bill"}
)

fig.show()

## 13. Alternative: Use `px.histogram()` for Aggregated Bar Charts

### Request
Create an average bar chart directly using Plotly.

### Explanation
`px.bar()` does not have `histfunc`.
For automatic aggregation, use `px.histogram()` with `histfunc="avg"`.

In [15]:
fig = px.histogram(
    data_frame=df,
    x="day",
    y="total_bill",
    color="smoker",
    barmode="group",
    histfunc="avg",
    title="Average Total Bill by Day and Smoker using px.histogram"
)

fig.show()

## 14. Box Plot: Tip Distribution by Day

### Request
Create a box plot showing `tip` distribution for each day.

### Explanation
Box plots show:
- median
- spread
- outliers
- differences between groups

In [16]:
fig = px.box(
    data_frame=df,
    x="day",
    y="tip",
    color="day",
    title="Tip Distribution by Day"
)

fig.show()

## 15. Box Plot with Two Categories

### Request
Compare `total_bill` by `smoker`, separated by `sex`.

### Explanation
This shows how one numerical column changes across two categorical variables.

In [17]:
fig = px.box(
    data_frame=df,
    x="smoker",
    y="total_bill",
    color="sex",
    title="Total Bill by Smoker Status and Sex"
)

fig.show()

## 16. Histogram: Distribution of Total Bill

### Request
Create a histogram of `total_bill`.

### Explanation
Histograms show how a numerical variable is distributed.

In [18]:
fig = px.histogram(
    data_frame=df,
    x="total_bill",
    nbins=25,
    title="Distribution of Total Bill"
)

fig.show()

## 17. Histogram with Category Color

### Request
Show the distribution of `tip`, colored by `time`.

### Explanation
This compares lunch and dinner tip distributions.

In [19]:
fig = px.histogram(
    data_frame=df,
    x="tip",
    color="time",
    nbins=20,
    barmode="overlay",
    title="Tip Distribution by Time"
)

fig.show()

## 18. Create a New Column: Tip Percentage

### Request
Create a new column called `tip_percent`.

### Explanation
Tip percentage is often more useful than raw tip amount.

Formula:

`tip_percent = tip / total_bill * 100`

In [20]:
df["tip_percent"] = (df["tip"] / df["total_bill"]) * 100

df[["total_bill", "tip", "tip_percent"]].head()

,total_bill,tip,tip_percent
0,16.99,1.01,5.944673
1,10.34,1.66,16.054159
2,21.01,3.50,16.658734
3,23.68,3.31,13.978041
4,24.59,3.61,14.680765


## 19. Line Plot: Tip Percentage by Row Index

### Request
Plot `tip_percent` using the dataframe index.

### Explanation
A line plot can show how values change across ordered rows.
Here, row index is not time, so this is only for practice.

In [21]:
fig = px.line(
    data_frame=df,
    x=df.index,
    y="tip_percent",
    title="Tip Percentage by Row Index",
    labels={"x": "Row Index", "tip_percent": "Tip Percentage"}
)

fig.show()

## 20. Sorted Line Plot: Total Bill vs Tip

### Request
Sort the dataframe by `total_bill`, then create a line plot between `total_bill` and `tip`.

### Explanation
Sorting makes the line plot easier to read.
Without sorting, the line jumps randomly between rows.

In [22]:
df_sorted = df.sort_values("total_bill")

fig = px.line(
    data_frame=df_sorted,
    x="total_bill",
    y="tip",
    title="Tip Trend as Total Bill Increases"
)

fig.show()

## 21. Bubble Chart

### Request
Create a scatter plot where bubble size represents party size.

### Explanation
A bubble chart is a scatter plot with an extra variable represented by point size.

In [23]:
fig = px.scatter(
    data_frame=df,
    x="total_bill",
    y="tip",
    size="size",
    color="day",
    hover_data=["sex", "smoker", "time"],
    title="Bubble Chart: Total Bill vs Tip"
)

fig.show()

## 22. Correlation Heatmap

### Request
Create a heatmap showing correlation between numerical columns.

### Explanation
Correlation shows the strength of relationship between numerical variables.

Values:
- close to 1: strong positive relationship
- close to -1: strong negative relationship
- close to 0: weak relationship

In [24]:
corr = df.corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=True,
    title="Correlation Heatmap"
)

fig.show()

## 23. 3D Scatter Plot

### Request
Create a 3D scatter plot using:
- `total_bill`
- `tip`
- `size`

### Explanation
3D plots allow us to visualize three numerical variables at once.
Use them carefully because they can become harder to read.

In [25]:
fig = px.scatter_3d(
    data_frame=df,
    x="total_bill",
    y="tip",
    z="size",
    color="day",
    title="3D Scatter Plot: Total Bill, Tip, and Party Size"
)

fig.show()

# Part B — Plotly Graph Objects

Graph Objects (`go`) gives more manual control than Plotly Express.

Use it when you want to:
- add multiple traces manually
- customize layout more deeply
- build charts step by step

## 24. Manual Scatter Plot with Graph Objects

### Request
Create a scatter plot manually using `go.Figure()` and `go.Scatter()`.

### Explanation
With Graph Objects, we manually create the figure and add traces.

In [26]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["total_bill"],
        y=df["tip"],
        mode="markers",
        name="Tips"
    )
)

fig.update_layout(
    title="Manual Scatter Plot: Total Bill vs Tip",
    xaxis_title="Total Bill",
    yaxis_title="Tip"
)

fig.show()

## 25. Graph Objects with Two Traces

### Request
Create one scatter trace for smokers and one for non-smokers.

### Explanation
This demonstrates why Graph Objects is useful:
we can manually control each group.

In [27]:
smokers = df[df["smoker"] == "Yes"]
non_smokers = df[df["smoker"] == "No"]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=smokers["total_bill"],
        y=smokers["tip"],
        mode="markers",
        name="Smoker"
    )
)

fig.add_trace(
    go.Scatter(
        x=non_smokers["total_bill"],
        y=non_smokers["tip"],
        mode="markers",
        name="Non-Smoker"
    )
)

fig.update_layout(
    title="Manual Scatter Plot by Smoker Status",
    xaxis_title="Total Bill",
    yaxis_title="Tip"
)

fig.show()

## 26. Manual Bar Chart with Graph Objects

### Request
Create a bar chart for average total bill per day using `go.Bar`.

### Explanation
We use the already aggregated dataframe from earlier.

In [28]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=mean_bill_by_day["day"],
        y=mean_bill_by_day["total_bill"],
        name="Average Total Bill"
    )
)

fig.update_layout(
    title="Manual Bar Chart: Average Total Bill per Day",
    xaxis_title="Day",
    yaxis_title="Average Total Bill"
)

fig.show()

# Part C — Mini Practice Section

These exercises combine what we learned without repeating the exact same technical solution.

## 27. Practice: Average Tip by Day

### Request
Calculate the average tip by day and create a bar chart.

### Explanation
This practices aggregation + bar chart.

In [29]:
avg_tip_by_day = (
    df.groupby("day", observed=False)["tip"]
      .mean()
      .reset_index()
)

fig = px.bar(
    data_frame=avg_tip_by_day,
    x="day",
    y="tip",
    title="Average Tip by Day",
    labels={"tip": "Average Tip"}
)

fig.show()

## 28. Practice: Which Day Has the Highest Average Tip Percentage?

### Request
Calculate the average `tip_percent` per day and sort from highest to lowest.

### Explanation
This combines feature engineering, grouping, and sorting.

In [30]:
avg_tip_percent_by_day = (
    df.groupby("day", observed=False)["tip_percent"]
      .mean()
      .reset_index()
      .sort_values("tip_percent", ascending=False)
)

avg_tip_percent_by_day

,day,tip_percent
1,Fri,16.991303
3,Sun,16.689729
0,Thur,16.127563
2,Sat,15.315172


In [31]:
fig = px.bar(
    data_frame=avg_tip_percent_by_day,
    x="day",
    y="tip_percent",
    title="Average Tip Percentage by Day",
    labels={"tip_percent": "Average Tip Percentage"}
)

fig.show()

## 29. Practice: Compare Lunch and Dinner Bills

### Request
Use a box plot to compare `total_bill` between Lunch and Dinner.

### Explanation
A box plot is useful when comparing distributions between groups.

In [32]:
fig = px.box(
    data_frame=df,
    x="time",
    y="total_bill",
    color="time",
    title="Total Bill Distribution: Lunch vs Dinner"
)

fig.show()

## 30. Practice: Relationship Between Tip Percentage and Total Bill

### Request
Create a scatter plot between `total_bill` and `tip_percent`.

### Explanation
This helps answer:
Do people give a higher percentage tip when the bill is larger?

In [34]:
# pip install statsmodels  for using trendline = ols
fig = px.scatter(
    data_frame=df,
    x="total_bill",
    y="tip_percent",
    color="smoker",
    trendline="ols",
    title="Total Bill vs Tip Percentage"
)

fig.show()

# Final Notes

## Key Takeaways

- Use `px.scatter()` for relationships between numerical variables.
- Use `px.bar()` when data is already aggregated.
- Use `px.histogram(..., histfunc="avg")` when you want Plotly to aggregate automatically.
- Use `px.box()` to compare distributions across categories.
- Use `px.imshow()` for heatmaps.
- Use `go.Figure()` when you need manual control.
- Always understand your data before plotting.